# ForgeGuard: Receipt or Deceit
### A Cross-Architecture Analysis of Convolutional Neural Network Models in Detecting Forged Digital Transaction Receipts
**Notre Dame of Midsayap College (NDMC) — BSCS Thesis Writing 1**  
**Researchers**: Rogie P. Bacanto, Daniela S. Ungab  
**Adviser**: Ms. Doris Ann Mariano  

---
### Experimental Protocol (Balanced 50/50 Quota + Unseen Generalization Holdout)
This notebook trains and empirically evaluates three CNN architectures on a **pure 1:1 balanced dataset (228 Authentic vs. 228 Stratified Forged)** preprocessed with **Error Level Analysis (ELA 90Q / 15x)**:
1. **Basic CNN**: Custom 3-layer sequential network (~2.1M params)
2. **MobileNetV2**: Lightweight inverted residual depthwise separable CNN (~3.4M params)
3. **ResNet50**: 50-layer deep residual network with bottleneck blocks (~23.5M params)

**Scientific Rigor**: The remaining **621 forged receipts** are strictly held back as an **Unseen Out-of-Distribution Stress Test** across 7 forgery categories.

In [ ]:
# Step 1: Clone the thesis repository containing dataset & preprocessors
!git clone https://github.com/DeathKnell837/NDMC-BSCS-THESIS-PREP.git
%cd NDMC-BSCS-THESIS-PREP/thesis-system
!pip install -q Pillow numpy scipy scikit-learn matplotlib seaborn

In [ ]:
# Step 2: Verify GPU & import libraries
import os, glob, time, json, shutil
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models, applications
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU Available:", gpus if gpus else "Running on CPU")

In [ ]:
# Step 3: Load Balanced Dataset (228 Authentic vs. 228 Stratified Forged) + 621 Holdout Fakes
from preprocessing.ela import compute_ela

IMG_SIZE = (128, 128)
IMAGE_EXTENSIONS = ("*.jpg", "*.jpeg", "*.png", "*.webp")

auth_dir = "dataset/authentic/compressed"
forged_dir = "dataset/forged/compressed"

# 1. Load all 228 Authentic Receipts (Label: 0)
auth_files = []
for ext in IMAGE_EXTENSIONS:
    auth_files.extend(glob.glob(os.path.join(auth_dir, ext)))
    auth_files.extend(glob.glob(os.path.join(auth_dir, ext.upper())))
auth_files = sorted(list(set(auth_files)))
print(f"[1/3] Found {len(auth_files)} Authentic receipts.")

# 2. Stratified Sampling of exactly 228 Forged Receipts (114 Edited + 114 Generated)
quotas = {
    "amount_alteration": 29,
    "name_modification": 29,
    "ref_fabrication": 29,
    "font_tampering": 27,
    "ai_generated_template": 81,
    "ai_diffusion_generated": 25,
    "full_template": 8
}

balanced_forged_files = []
holdout_forged_files = []
for subcat, q in quotas.items():
    sub_files = []
    for ext in IMAGE_EXTENSIONS:
        sub_files.extend(glob.glob(os.path.join(forged_dir, subcat, ext)))
        sub_files.extend(glob.glob(os.path.join(forged_dir, subcat, ext.upper())))
    sub_files = sorted(list(set(sub_files)))
    balanced_forged_files.extend(sub_files[:q])
    holdout_forged_files.extend(sub_files[q:])
    print(f"      - {subcat:25s}: Selected={min(q, len(sub_files)):2d} | Holdout Reserve={max(0, len(sub_files)-q):3d}")

print(f"[2/3] Balanced Training Set: {len(auth_files)} Authentic vs. {len(balanced_forged_files)} Forged (Total: {len(auth_files)+len(balanced_forged_files)})")
print(f"[3/3] Holdout Stress-Test Set: {len(holdout_forged_files)} Unseen Forgeries")

# Preprocess Balanced Dataset using ELA
X, y = [], []
print("\nExtracting ELA features for Balanced Dataset...")
for f in auth_files:
    with Image.open(f) as img:
        ela = compute_ela(img, quality=90, scale=15.0).resize(IMG_SIZE)
        X.append(np.array(ela, dtype=np.float32) / 255.0)
        y.append(0)

for f in balanced_forged_files:
    with Image.open(f) as img:
        ela = compute_ela(img, quality=90, scale=15.0).resize(IMG_SIZE)
        X.append(np.array(ela, dtype=np.float32) / 255.0)
        y.append(1)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

# Preprocess Holdout Stress-Test Set using ELA
print("Extracting ELA features for Unseen Holdout Set...")
X_holdout = []
for f in holdout_forged_files:
    with Image.open(f) as img:
        ela = compute_ela(img, quality=90, scale=15.0).resize(IMG_SIZE)
        X_holdout.append(np.array(ela, dtype=np.float32) / 255.0)
X_holdout = np.array(X_holdout, dtype=np.float32)
y_holdout = np.ones(len(X_holdout), dtype=np.int32)

# Stratified Train (70%) / Val (15%) / Test (15%) Split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"\nDataset Ready! Train: {len(X_train)} (Auth:{np.sum(y_train==0)}, Fake:{np.sum(y_train==1)}) | Val: {len(X_val)} | Test: {len(X_test)} (Auth:{np.sum(y_test==0)}, Fake:{np.sum(y_test==1)})")
print(f"Unseen Stress Test Samples: {len(X_holdout)} fakes")

In [ ]:
# Step 4: Define the 3 CNN Architectures
def build_basic_cnn():
    model = models.Sequential([
        layers.Input(shape=(128, 128, 3)),
        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def build_mobilenetv2():
    base = applications.MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights="imagenet")
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss="binary_crossentropy", metrics=["accuracy"])
    return model

def build_resnet50():
    base = applications.ResNet50(input_shape=(128, 128, 3), include_top=False, weights="imagenet")
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss="binary_crossentropy", metrics=["accuracy"])
    return model

print("All 3 model architectures compiled successfully!")

In [ ]:
# Step 5: Train, Benchmark, and Evaluate on Both Balanced Test Set & Unseen Stress Test
os.makedirs("models", exist_ok=True)

models_dict = {
    "Basic_CNN": build_basic_cnn(),
    "MobileNetV2": build_mobilenetv2(),
    "ResNet50": build_resnet50()
}

results = {}
histories = {}

for name, model in models_dict.items():
    print(f"\n{'='*25} Training {name} {'='*25}")
    t0 = time.time()
    histories[name] = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=20,
        batch_size=16,
        verbose=1
    )
    train_time = time.time() - t0
    
    # 1. Primary Balanced Test Evaluation
    t_inf = time.time()
    y_prob = model.predict(X_test, verbose=0)
    lat_ms = ((time.time() - t_inf) / len(X_test)) * 1000.0
    y_pred = (y_prob >= 0.5).astype(int).flatten()
    
    acc = float(accuracy_score(y_test, y_pred))
    prec = float(precision_score(y_test, y_pred, zero_division=0))
    rec = float(recall_score(y_test, y_pred, zero_division=0))
    f1 = float(f1_score(y_test, y_pred, zero_division=0))
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # 2. Unseen Holdout Stress Test Evaluation (621 Unseen Fakes)
    y_holdout_prob = model.predict(X_holdout, verbose=0)
    y_holdout_pred = (y_holdout_prob >= 0.5).astype(int).flatten()
    stress_recall = float(recall_score(y_holdout, y_holdout_pred, zero_division=0))
    stress_caught = int(np.sum(y_holdout_pred == 1))
    
    results[name] = {
        "architecture": name.replace("_", " "),
        "condition": "Standard",
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "latency_ms": float(lat_ms),
        "train_duration_s": float(train_time),
        "test_size": len(X_test),
        "authentic_test_count": int(np.sum(y_test == 0)),
        "forged_test_count": int(np.sum(y_test == 1)),
        "confusion": {
            "tp": int(tp),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn)
        },
        "stress_test": {
            "holdout_total": len(X_holdout),
            "fakes_caught": stress_caught,
            "detection_rate": stress_recall
        }
    }
    
    model_filename = f"models/{name.lower()}.keras"
    model.save(model_filename)
    print(f"\nSaved {model_filename}")
    print(f"Balanced Test Acc: {acc*100:.2f}% | Prec: {prec*100:.2f}% | Rec: {rec*100:.2f}% | F1: {f1:.4f} | Lat: {lat_ms:.2f}ms")
    print(f"Unseen Stress Test: Caught {stress_caught}/{len(X_holdout)} unseen fakes ({stress_recall*100:.2f}% detection rate)")

# Save final evaluation metrics
with open("models/evaluation_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved models/evaluation_metrics.json successfully!")

In [ ]:
# Step 6: Print Thesis Summary Table & Download Bundle
print("="*80)
print("FORGEGUARD: OFFICIAL BALANCED EVALUATION BENCHMARK")
print("="*80)
print(f"{'Architecture':15s} | {'Accuracy':9s} | {'Precision':9s} | {'Recall':9s} | {'F1-Score':9s} | {'Latency':10s} | {'Stress Rec':10s}")
print("-"*80)
for name, v in results.items():
    arch = v["architecture"]
    acc = f"{v['accuracy']*100:.2f}%"
    prec = f"{v['precision']*100:.2f}%"
    rec = f"{v['recall']*100:.2f}%"
    f1 = f"{v['f1_score']:.4f}"
    lat = f"{v['latency_ms']:.2f} ms"
    stress = f"{v['stress_test']['detection_rate']*100:.2f}%"
    print(f"{arch:15s} | {acc:9s} | {prec:9s} | {rec:9s} | {f1:9s} | {lat:10s} | {stress:10s}")
print("="*80)

# Package all models and metrics into a single zip file
zip_filename = "forgeguard_balanced_models"
shutil.make_archive(zip_filename, "zip", "models")
print(f"\nZip archive created: {zip_filename}.zip")

# Trigger Google Colab Download
try:
    from google.colab import files
    files.download(f"{zip_filename}.zip")
    print("Download started! Extract the zip and put the files into your thesis-system/models/ directory.")
except Exception as e:
    print(f"Direct download not available outside Colab. File is saved at: {zip_filename}.zip")